# Simulation checkpointing command in XR headset

## Simulation and server setup

We start with the usual server setup, using a pre-bundled OpenMM simulation of a methane + nanotube system:

In [1]:
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

nanotube_simulation = OpenMMSimulation.from_xml_path("../systems/openmm/nanotube.openmm.zip")

imd_runner = OmniRunner.with_basic_server(nanotube_simulation, name='simulation checkpoints example')
imd_runner.load(0)
imd_runner.print_basic_info()

Serving "vr-commands-server" (ws://localhost:38801), discoverable on all interfaces on port 54545
Available simulations:
[0]: "nanotube.openmm"
Switched to [0]: "nanotube.openmm"


## Defining a checkpointing command

We'll define a command that, when triggered, saves the current state of the current simulation and adds to the list of simulations in simulation selection:

In [4]:
def clone_omni_omm(omni_omm):
    """
    Create an independent copy of a given NanoVer OpenMMSimulation.
    """
    from nanover.openmm import serializer

    # get the underlying openmm simulation object
    omm = omni_omm.simulation

    # clone it by serializing and then deserializing
    data = serializer.serialize_simulation(omm)
    omm_clone = serializer.deserialize_simulation(data)

    # wrap the cloned openmm simulation in OpenMMSimulation
    omni_omm_clone = OpenMMSimulation.from_simulation(omm_clone)
    
    # give it different name
    omni_omm_clone.name = omni_omm.name + "'"
    
    return omni_omm_clone


def add_checkpoint_simulation():
    imd_runner.add_simulation(clone_omni_omm(imd_runner.simulation))


# add the command to the server,
# commands with names beginning with "user/" are displayed in the VR client
imd_runner.app_server.register_command("user/checkpoint", add_checkpoint_simulation, icon="🚩")

## Using the command

The command is now available in the [NanoVer iMD-XR client](https://irl2.github.io/nanover-docs/installation.html#installing-the-imd-xr-client). Connect to the server and use the "User Commands" menu to trigger the command; it will add a new listing to the simulations selection menu, just like in this video:

<video controls src="../figures/imd-vr-checkpoint-command-small.mp4" alt="video demonstrating the command in vr" />